# 01 — Generate open host-mass catalogs with **TNG median-SFMS quenching** ($\log M_{200c} > 12$)

Same as `01_generate_catalogs_massive.ipynb`, but satellites are flagged **quenched relative to the
simulation's own median star-forming main sequence (SFMS)**, fit *per snapshot* — **not** the fixed
SDSS relation. For TNG100 & TNG50 at z=0 and z=0.05, for **all central hosts with
$\log_{10} M_{200c} > 12$**, it computes each satellite's azimuthal angle $\alpha\in[0,90]$ and its
3D distance from the central; **no radius cut**.

**Quench definition (the change):**
* Fit the SFMS as the running median of $\log_{10}\mathrm{SFR}$ vs $\log_{10}M_*$ for **all galaxies**
  in the snapshot with $M_* > 10^{8}\,M_\odot$ and $\mathrm{SFR}>0$ (matching the A&A 2025 recipe).
* A satellite is **quenched** if its $\log_{10}\mathrm{SFR}$ is $\ge 1$ dex below that median SFMS
  (`QUENCH_DEX`). $\mathrm{SFR}=0$ satellites are quenched by construction.
* The catalog's `quenched` column now uses this **TNG SFMS**; the old fixed-SDSS flag is kept as
  `quenched_sdss` for a side-by-side comparison.

**Selections:** hosts $\log_{10} M_{200c} > 12$ with reliable g+i Sersic morphology; satellites
$M_* > 10^7\,M_\odot$ (so any floor is a post-selection). Both host and satellite masses are stored,
so any host window (12.0-12.5, 13.0-14.0, …) is reproducible downstream.

> **Redshift note.** "z = 0.5" is interpreted as **z = 0.05** (the SDSS-SKIRT-mock redshift).
> **Output:** writes the **same** `tng_satellites_hostlogM12.0plus_logM7.00.csv` as the SDSS version
> (now with TNG-based `quenched` + `quenched_sdss`), so the analysis notebooks pick it up unchanged.
> **Runs only on Binder** (raw TNG data). Masses are physical $M_\odot$; little-h $=0.6774$.

In [11]:
import os
import numpy as np
import pandas as pd
import h5py
import illustris_python as il

## Configuration

Loop over both simulations and both redshifts. In addition to the host/satellite cuts, the SFMS
block controls the quench definition: the SFMS is fit from every galaxy with $M_*>$ `SFMS_LOGM_MIN`
and $\mathrm{SFR}>0$ in the snapshot, and quenched means `QUENCH_DEX` below it.

In [12]:
# simulations and redshifts to build (the notebook loops over the full product)
SIMS      = ["TNG100", "TNG50"]
REDSHIFTS = ["z0", "z0p05"]                 # z0 -> snap 99 (z=0); z0p05 -> snap 98 (z=0.05)

_SIM_DIR  = {"TNG100": "L75n1820TNG", "TNG50": "L35n2160TNG"}
_LBOX_MPC_H = {"TNG100": 75.0, "TNG50": 35.0}
_SNAP     = {"z0": 99, "z0p05": 95}
_ZVAL     = {"z0": 0.0, "z0p05": 0.0485}    # redshift -> physical distance scale factor

# raw data root on Binder: ~/SimulationData/<sim_dir>
TNG_PARENT = os.path.expanduser("~/SimulationData")

h = 0.6774                                  # IllustrisTNG little-h (Planck 2015)

# --- selections (physical M_sun) ---
HOST_LOGM200         = (12.0, None)     # (min, max) log10 M200c host window; None = open end. Here: > 12, no upper limit
CENTRAL_LOGMSTAR_MIN = 0.0                  # optional extra floor on central M* (0 = off; host cut is halo mass)
SAT_LOGM_MIN         = 7.0                  # keep satellites with log10 M*_sat > this
SN_MIN               = 2.5                  # SDSS-Sersic S/N-per-pixel reliability floor

# --- quench definition: 1 dex below the SIMULATION's own median SFMS (fit per snapshot) ---
SFMS_LOGM_MIN  = 8.0            # fit the SFMS from all galaxies with log10 M* > this and SFR > 0
SFMS_FIT_RANGE = (8.0, 10.5)    # log10 M* range over which to fit the running-median line (SF-dominated)
SFMS_BIN       = 0.25          # log10 M* bin width for the running median
QUENCH_DEX     = 1.0           # quenched if log10 SFR is this many dex below the median SFMS

# NO radius cut here: satellites at all radii are written (downstream toggles its own cut).
# The catalog stores d_3d_kpc and d_r200_3d so any aperture can be applied later.

DATA_ROOT = "../data2"
_lo, _hi  = HOST_LOGM200
if   _lo is not None and _hi is not None: HOST_TAG = f"hostlogM{_lo:.1f}-{_hi:.1f}"
elif _lo is not None:                     HOST_TAG = f"hostlogM{_lo:.1f}plus"      # open upper end
elif _hi is not None:                     HOST_TAG = f"hostlogMto{_hi:.1f}"        # open lower end
else:                                     HOST_TAG = "allcentrals"
CAT_TAG   = f"logM{SAT_LOGM_MIN:.2f}"       # e.g. 'logM7.00'
CAT_FILE  = f"tng_satellites_{HOST_TAG}_{CAT_TAG}.csv"   # e.g. tng_satellites_hostlogM12.0plus_logM7.00.csv
print("build:", SIMS, "x", REDSHIFTS, "| host window", HOST_LOGM200,
      "| sat M* >", SAT_LOGM_MIN, "->", CAT_FILE)

build: ['TNG100', 'TNG50'] x ['z0', 'z0p05'] | host window (12.0, None) | sat M* > 7.0 -> tng_satellites_hostlogM12.0plus_logM7.00.csv


## Helpers

Same geometry as `notebooks/01`. Two quench helpers: `quenched_flag_sdss` (the fixed SDSS relation,
kept for comparison) and `fit_median_sfms`, which fits the simulation's own median SFMS from the
snapshot galaxies — the new quench boundary is 1 dex below that line.

In [13]:
def host_morph_to_satellites(sdss, first_sub, n_subs, subfind_ids, n_sub):
    """Look up each host's Sersic morphology and broadcast it to its satellites."""
    theta = np.zeros(n_sub); flag = np.zeros(n_sub); sflag = np.zeros(n_sub); sn = np.zeros(n_sub)
    theta_all = np.asarray(sdss["sersic_theta"]) * 180.0 / np.pi
    flag_all  = np.asarray(sdss["flag"]); sflag_all = np.asarray(sdss["flag_sersic"])
    sn_all    = np.asarray(sdss["sn_per_pixel"])
    for k in range(len(first_sub)):
        c = first_sub[k]
        row = np.where(subfind_ids == c)[0]
        if row.size == 0:                                  # host has no SKIRT entry -> flag bad
            flag[c + 1:c + n_subs[k]] = 1
            continue
        r = row[0]; sl = slice(c + 1, c + n_subs[k])
        theta[sl] = theta_all[r]; flag[sl] = flag_all[r]; sflag[sl] = sflag_all[r]; sn[sl] = sn_all[r]
    return theta, flag, sflag, sn

def wrap_min_image(d, L):
    """Minimum-image periodic wrap of a separation vector."""
    return d - L * np.round(d / L)

def angle_from_major_axis(phi_sat_deg, pa_host_deg):
    """Fold |phi_sat - PA_host| into [0, 90] using the disk 180-deg / mirror symmetry."""
    phi = np.mod(phi_sat_deg, 360.0)
    d1 = np.abs(phi - np.mod(pa_host_deg, 360.0))
    d2 = np.abs(phi - np.mod(pa_host_deg + 180.0, 360.0))
    d1 = np.minimum(d1, 360.0 - d1); d2 = np.minimum(d2, 360.0 - d2)
    m = np.minimum(d1, d2)
    return np.minimum(m, 180.0 - m)

def quenched_flag_sdss(logmstar_phys, sfr):
    """1 if >= 1 dex below the FIXED SDSS SFMS (Martin-Navarro+ 2021), else 0. (kept for comparison)"""
    sfr_ms = 10.0 ** (0.75 * logmstar_phys - 7.5)
    return (sfr < sfr_ms / 10.0).astype(int)

def fit_median_sfms(logmstar, sfr, logm_min, fit_range, binw, min_per_bin=20):
    """Linear fit to the running median of log10(SFR) vs log10(M*) for the SF (SFR>0) galaxies.

    Returns (slope, intercept, centers, median_logsfr). This IS the simulation's own SFMS:
    log10 SFR_MS = slope*log10 M* + intercept. Falls back to the SDSS coefficients if too few
    star-forming galaxies are available to fit.
    """
    sf = (logmstar > logm_min) & (sfr > 0)
    lm = np.asarray(logmstar)[sf]; ls = np.log10(np.asarray(sfr)[sf])
    edges = np.arange(fit_range[0], fit_range[1] + 1e-9, binw)
    cents = 0.5 * (edges[:-1] + edges[1:])
    med = np.full(len(cents), np.nan)
    idx = np.digitize(lm, edges) - 1
    for j in range(len(cents)):
        m = idx == j
        if m.sum() >= min_per_bin:
            med[j] = np.median(ls[m])
    ok = np.isfinite(med)
    if ok.sum() >= 2:
        slope, intercept = np.polyfit(cents[ok], med[ok], 1)
    else:
        slope, intercept = 0.75, -7.5                      # fallback: SDSS relation
    return slope, intercept, cents, med

## Build one catalog per (simulation, redshift)

`process` loads the snapshot, **fits that snapshot's median SFMS** from all galaxies
($M_*>$`SFMS_LOGM_MIN`, $\mathrm{SFR}>0$), selects the central hosts in the `HOST_LOGM200` window
with reliable Sersic fits, flags their satellites, computes $\alpha$ and distances, defines
`quenched` as `QUENCH_DEX` below the TNG SFMS (with `quenched_sdss` alongside), and writes the
catalog. Each snapshot uses its own SFMS, so z=0 and z=0.05 are calibrated independently.

In [14]:
def process(sim, redshift):
    sim_dir = _SIM_DIR[sim]; snap = _SNAP[redshift]; A = 1.0 / (1.0 + _ZVAL[redshift])
    Lbox = _LBOX_MPC_H[sim] * 1000.0 / h                         # physical kpc
    root = os.path.join(TNG_PARENT, sim_dir)
    basePath = os.path.join(root, "output")
    sp = f"snapnum_{snap:03d}"
    morph_g = os.path.join(root, f"postprocessing/skirt_images/sdss/{sp}/morphs_g.hdf5")
    morph_i = os.path.join(root, f"postprocessing/skirt_images/sdss/{sp}/morphs_i.hdf5")
    subfind_id_path = os.path.join(root, f"postprocessing/skirt_images/sdss/{sp}/subfind_ids.txt")

    gc = os.path.join(basePath, f"groups_{snap:03d}", f"fof_subhalo_tab_{snap:03d}.0.hdf5")
    if not (os.path.exists(gc) and os.path.exists(morph_g)):
        print(f"[skip] {sim} {redshift}: data not found at {root}")
        return None

    groups = il.groupcat.loadHalos(basePath, snap,
        fields=["GroupFirstSub", "Group_M_Crit200", "Group_R_Crit200", "GroupNsubs"])
    subhalos = il.groupcat.loadSubhalos(basePath, snap,
        fields=["SubhaloGrNr", "SubhaloMassType", "SubhaloCM", "SubhaloSFR"])
    n_sub = len(subhalos["SubhaloGrNr"])
    sdss_g = h5py.File(morph_g, "r"); sdss_i = h5py.File(morph_i, "r")
    subfind_ids = np.loadtxt(subfind_id_path)

    # per-subhalo stellar masses (both conventions)
    with np.errstate(divide="ignore"):
        mstar_phys = np.log10(subhalos["SubhaloMassType"][:, 4] * 1e10 / h)      # physical M_sun
        mstar_hinv = np.log10(subhalos["SubhaloMassType"][:, 4] * 1e10)          # h^-1 M_sun
    sat_sfr = np.asarray(subhalos["SubhaloSFR"])

    # ---- fit THIS snapshot's median SFMS from all galaxies (M* > SFMS_LOGM_MIN, SFR > 0) ----
    sfms_m, sfms_b, _mc, _md = fit_median_sfms(mstar_phys, sat_sfr, SFMS_LOGM_MIN, SFMS_FIT_RANGE, SFMS_BIN)
    def _quenched_tng(logm, sfr):
        thr = 10.0 ** (sfms_m * np.asarray(logm) + sfms_b - QUENCH_DEX)   # QUENCH_DEX below TNG median SFMS
        return (np.asarray(sfr) < thr).astype(int)
    print(f"     {sim} {redshift} median SFMS: log SFR = {sfms_m:.3f} logM* {sfms_b:+.3f}"
          f"   (SDSS fixed: 0.750 logM* -7.500)")

    # ---- host (central) selection: centrals in the HOST_LOGM200 window + central M* floor ----
    with np.errstate(divide="ignore"):
        host_logm200_phys = np.log10(groups["Group_M_Crit200"] * 1e10 / h)
    first_all = groups["GroupFirstSub"]
    valid = first_all >= 0                                          # group has a central subhalo
    cen_mstar = np.full(len(first_all), -np.inf)
    cen_mstar[valid] = mstar_phys[first_all[valid]]
    host_sel = valid & (cen_mstar > CENTRAL_LOGMSTAR_MIN)
    lo, hi = HOST_LOGM200
    if lo is not None: host_sel &= host_logm200_phys > lo
    if hi is not None: host_sel &= host_logm200_phys < hi

    first_sub = groups["GroupFirstSub"][host_sel]
    n_subs    = groups["GroupNsubs"][host_sel]
    host_m200_phys_all = host_logm200_phys[host_sel]
    host_m200_hinv_all = np.log10(groups["Group_M_Crit200"][host_sel] * 1e10)
    cen_mstar_all      = cen_mstar[host_sel]

    # per-satellite arrays, filled only for satellites of selected centrals
    sat_mask       = np.zeros(n_sub, dtype=bool)
    host_id_arr    = np.full(n_sub, -1, dtype=int)
    host_center    = np.zeros((n_sub, 3))
    host_r200_arr  = np.zeros(n_sub)
    host_m200_phys = np.zeros(n_sub); host_m200_hinv = np.zeros(n_sub)
    host_mstar_arr = np.zeros(n_sub)
    for k in range(len(first_sub)):
        c = first_sub[k]; sl = slice(c + 1, c + n_subs[k])
        sat_mask[sl] = True
        host_id_arr[sl] = k
        host_center[sl] = subhalos["SubhaloCM"][c] / h
        host_r200_arr[sl] = groups["Group_R_Crit200"][host_sel][k] / h
        host_m200_phys[sl] = host_m200_phys_all[k]; host_m200_hinv[sl] = host_m200_hinv_all[k]
        host_mstar_arr[sl] = cen_mstar_all[k]

    # ---- host Sersic orientation + quality (mean of g and i) ----
    tg, fg, sfg, sng = host_morph_to_satellites(sdss_g, first_sub, n_subs, subfind_ids, n_sub)
    ti, fi, sfi, sni = host_morph_to_satellites(sdss_i, first_sub, n_subs, subfind_ids, n_sub)
    host_theta = 0.5 * (tg + ti)
    host_good  = ((fg == 0) & (sfg == 0) & (sng > SN_MIN) &
                  (fi == 0) & (sfi == 0) & (sni > SN_MIN))

    # ---- azimuthal angle alpha and 3D / projected host-centric distances ----
    rel = wrap_min_image(subhalos["SubhaloCM"] / h - host_center, Lbox)          # comoving kpc
    phi_2d = np.degrees(np.arctan2(rel[:, 1], rel[:, 0]))
    alpha = angle_from_major_axis(phi_2d, host_theta)
    _d3d = np.linalg.norm(rel, axis=1); _dpr = np.hypot(rel[:, 0], rel[:, 1])
    d_3d_kpc = _d3d * A; d_proj_kpc = _dpr * A
    with np.errstate(invalid="ignore", divide="ignore"):
        gr = host_r200_arr > 0
        d_r200_3d   = np.where(gr, _d3d / host_r200_arr, np.nan)
        d_r200_proj = np.where(gr, _dpr / host_r200_arr, np.nan)

    # ---- select satellites (mass floor only; NO radius cut) and write ----
    sel = sat_mask & host_good & (mstar_phys > SAT_LOGM_MIN)
    df = pd.DataFrame({
        "host_id":          host_id_arr[sel],
        "mstar_phys":       mstar_phys[sel],
        "mstar_hinv":       mstar_hinv[sel],
        "host_mstar_phys":  host_mstar_arr[sel],       # central (host) stellar mass, log10 M_sun
        "host_m200_phys":   host_m200_phys[sel],
        "host_m200_hinv":   host_m200_hinv[sel],
        "sfr":              sat_sfr[sel],
        "alpha":            alpha[sel],
        "d_3d_kpc":         d_3d_kpc[sel],
        "d_proj_kpc":       d_proj_kpc[sel],
        "d_r200_3d":        d_r200_3d[sel],
        "d_r200_proj":      d_r200_proj[sel],
        "quenched":         _quenched_tng(mstar_phys[sel], sat_sfr[sel]),          # TNG median-SFMS based
        "quenched_sdss":    quenched_flag_sdss(mstar_phys[sel], sat_sfr[sel]),     # fixed SDSS SFMS (comparison)
    })
    assert df.alpha.between(0, 90).all(), "alpha outside [0, 90]!"

    outdir = os.path.join(DATA_ROOT, sim.lower(), redshift)
    os.makedirs(outdir, exist_ok=True)
    out = os.path.join(outdir, CAT_FILE)
    df.to_csv(out, index=False)
    print(f"[ok] {sim:6s} {redshift:5s}: hosts={host_sel.sum():6d}  "
          f"hosts_w_sat={df.host_id.nunique():5d}  sats={len(df):6d}  "
          f"f_q(tng)={df.quenched.mean():.3f} f_q(sdss)={df.quenched_sdss.mean():.3f}  -> {out}")
    sdss_g.close(); sdss_i.close()
    return dict(sim=sim, z=redshift, n_host_sel=int(host_sel.sum()),
                n_host=int(df.host_id.nunique()), n_sat=len(df),
                sfms_slope=float(sfms_m), sfms_intercept=float(sfms_b),
                fq_tng=float(df.quenched.mean()), fq_sdss=float(df.quenched_sdss.mean()), path=out)

## Run the full product (both simulations, both redshifts)

In [15]:
summary = []
for sim in SIMS:
    for z in REDSHIFTS:
        try:
            r = process(sim, z)
            if r: summary.append(r)
        except Exception as e:
            print(f"[error] {sim} {z}: {type(e).__name__}: {e}")

print("\n=== summary (TNG median-SFMS quenching) ===")
print(f"{'sim':6s} {'z':5s} {'sats':>7s}  {'SFMS slope/intercept':>22s}  {'f_q(tng)':>8s} {'f_q(sdss)':>9s}")
for r in summary:
    print(f"{r['sim']:6s} {r['z']:5s} {r['n_sat']:>7d}  "
          f"log SFR={r['sfms_slope']:.3f} logM*{r['sfms_intercept']:+.3f}  "
          f"{r['fq_tng']:>8.3f} {r['fq_sdss']:>9.3f}")

[ok] TNG100 z0   : hosts=  1890  hosts_w_sat= 1550  sats= 36252  f_q=0.772  -> ../data2/tng100/z0/tng_satellites_hostlogM12.0plus_logM7.00.csv
[ok] TNG100 z0p05: hosts=  1919  hosts_w_sat= 1564  sats= 39034  f_q=0.774  -> ../data2/tng100/z0p05/tng_satellites_hostlogM12.0plus_logM7.00.csv
[ok] TNG50  z0   : hosts=   207  hosts_w_sat=  165  sats=  5929  f_q=0.776  -> ../data2/tng50/z0/tng_satellites_hostlogM12.0plus_logM7.00.csv
[ok] TNG50  z0p05: hosts=   206  hosts_w_sat=  168  sats=  5560  f_q=0.765  -> ../data2/tng50/z0p05/tng_satellites_hostlogM12.0plus_logM7.00.csv

=== summary ===
TNG100 z0     hosts=  1890  hosts_w_sat= 1550  sats= 36252
TNG100 z0p05  hosts=  1919  hosts_w_sat= 1564  sats= 39034
TNG50  z0     hosts=   207  hosts_w_sat=  165  sats=  5929
TNG50  z0p05  hosts=   206  hosts_w_sat=  168  sats=  5560
